In [ ]:
# %%
import os
import re
import traceback
import pandas as pd
import cobra
from cobra.flux_analysis import pfba, gimme

# ======================= Paths =======================
MODEL_FILE = r"../models/iJO1366.xml"
INPUT_DIR = r"../data/mapped-gene-second"
OUTPUT_DIR = r"../data/gimme_single_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ======================= Parameters =======================
DOWN_THRESHOLD = -1.5
UP_THRESHOLD = 2

GIMME_THRESHOLD = 0.5
GIMME_FRACTION = 0.1

# ======================= Load model =======================
print("🚀 Loading model...")
model = cobra.io.read_sbml_model(MODEL_FILE)

try:
    model.solver = "gurobi"
    print("✅ Using Gurobi")
except:
    print("⚠️ Using default solver")

# biomass
biomass_rxn = [r.id for r in model.reactions if "biomass" in r.id.lower()][0]
print("Biomass:", biomass_rxn)

# baseline
baseline = model.optimize()
print("Baseline status:", baseline.status)

baseline_flux = pd.Series(baseline.fluxes, name="Flux_baseline")

# Reaction info (pre-cached for speed)
reaction_map = {
    r.id: (r.name, r.subsystem)
    for r in model.reactions
}

# ======================= GPR =======================
def gene_to_rxn_state(model, gene_state):
    num = {"high": 2, "medium": 1, "low": 0}
    gene_scores = {g: num[s] for g, s in gene_state.items()}

    rxn_state = {}

    for r in model.reactions:
        vals = [gene_scores.get(g.id.lower()) for g in r.genes if g.id.lower() in gene_scores]

        if not vals:
            rxn_state[r.id] = "medium"
        else:
            score = max(vals)
            if score >= 1.75:
                rxn_state[r.id] = "high"
            elif score <= 0.25:
                rxn_state[r.id] = "low"
            else:
                rxn_state[r.id] = "medium"

    return rxn_state


def rxn_to_gimme(rxn_state):
    return {
        r: (1.0 if s == "high" else 0.0 if s == "low" else 0.5)
        for r, s in rxn_state.items()
    }

# ======================= Single-file processing =======================
def process_file(fname):
    try:
        print(f"\n📂 Processing: {fname}")

        path = os.path.join(INPUT_DIR, fname)

        # Auto-detect the delimiter (fully fixes the earlier problem)
        df = pd.read_csv(path, sep=None, engine="python")

        print("列名:", df.columns.tolist())

        # ===== Clean column names =====
        df.columns = df.columns.str.strip()

        # ===== Auto-detect columns =====
        gene_col = df.columns[0]
        value_col = df.columns[1]

        # ===== Normalize genes =====
        df[gene_col] = df[gene_col].astype(str).str.strip().str.lower()

        df = df[df[gene_col].str.startswith("b")]

        # ===== fitness =====
        df["fitness"] = pd.to_numeric(df[value_col], errors="coerce")
        df = df.dropna(subset=["fitness"])

        print("有效基因数:", len(df))

        if len(df) == 0:
            print("❌ 无有效数据 → 跳过")
            return

        # ===== gene → state =====
        gene_state = {}
        for _, row in df.iterrows():
            g = row[gene_col]
            f = row["fitness"]

            if f <= DOWN_THRESHOLD:
                gene_state[g] = "high"
            elif f >= UP_THRESHOLD:
                gene_state[g] = "low"
            else:
                gene_state[g] = "medium"

        # ===== Match rate =====
        model_genes = set(g.id.lower() for g in model.genes)
        input_genes = set(gene_state.keys())
        common = model_genes & input_genes

        print(f"匹配成功: {len(common)}/{len(input_genes)}")

        if len(common) == 0:
            print("❌ 完全未匹配 → 跳过")
            return

        # ===== Copy the model =====
        local_model = model.copy()

        # ===== GIMME =====
        rxn_state = gene_to_rxn_state(local_model, gene_state)
        gimme_scores = rxn_to_gimme(rxn_state)

        try:
            print("🚀 Running GIMME...")
            sol = gimme(
                local_model,
                gimme_scores,
                objective=biomass_rxn,
                threshold=GIMME_THRESHOLD,
                fraction_of_optimum=GIMME_FRACTION
            )
            print("✅ GIMME success")

        except Exception as e:
            print("⚠️ GIMME failed → fallback pFBA:", e)
            sol = pfba(local_model)

        if sol is None:
            print("❌ solution None")
            return

        # ===== flux =====
        flux = pd.Series(sol.fluxes, name="Flux_condition")

        if flux.abs().sum() == 0:
            print("⚠️ flux 全0")

        # ===== Build output =====
        out = pd.DataFrame({
            "Reaction": [r.id for r in local_model.reactions]
        })

        out["Name"] = out["Reaction"].map(lambda x: reaction_map[x][0])
        out["Subsystem"] = out["Reaction"].map(lambda x: reaction_map[x][1])

        out = out.merge(
            baseline_flux.reset_index().rename(columns={"index": "Reaction"}),
            on="Reaction"
        )

        out = out.merge(
            flux.reset_index().rename(columns={"index": "Reaction"}),
            on="Reaction"
        )

        # ===== ΔFlux =====
        out["ΔFlux"] = out["Flux_condition"] - out["Flux_baseline"]

        # ===== State =====
        out["State_highmidlow"] = out["Reaction"].map(rxn_state)
        out["GIMME_score"] = out["Reaction"].map(gimme_scores)

        # ===== Growth =====
        out["Growth_Rate"] = sol.objective_value
        out["Solution_Status"] = sol.status

        print("ΔFlux统计:")
        print(out["ΔFlux"].describe())

        # ===== Filename (uppercase) =====
        name = os.path.splitext(fname)[0].upper()

        out_path = os.path.join(OUTPUT_DIR, f"GIMME_{name}.xlsx")
        out.to_excel(out_path, index=False)

        print("✅ Saved:", out_path)

    except Exception:
        print("❌ Error processing:", fname)
        traceback.print_exc()


# ======================= Main program =======================
if __name__ == "__main__":

    files = sorted([f for f in os.listdir(INPUT_DIR) if f.endswith(".csv")])

    print("Total files:", len(files))

    for i, f in enumerate(files):
        print(f"\n==== [{i+1}/{len(files)}] ====")
        process_file(f)

    print("\n🎉 ALL DONE")

🚀 Loading model...
⚠️ Using default solver
Biomass: BIOMASS_Ec_iJO1366_WT_53p95M
Baseline status: optimal
Total files: 323

==== [1/323] ====

📂 Processing: 16C.csv
列名: ['Gene', '16C - ']
有效基因数: 3825
匹配成功: 1246/3793
🚀 Running GIMME...
✅ GIMME success
ΔFlux统计:
count    2583.000000
mean       -0.058484
std         2.028293
min       -48.920719
25%         0.000000
50%         0.000000
75%         0.000000
max        35.169830
Name: ΔFlux, dtype: float64
✅ Saved: ../data/gimme_single_results\GIMME_16C.xlsx

==== [2/323] ====

📂 Processing: 18C.csv
列名: ['Gene', '18C - ']
有效基因数: 3821
匹配成功: 1244/3789
🚀 Running GIMME...
✅ GIMME success
ΔFlux统计:
count    2583.000000
mean        0.010428
std         0.996384
min       -11.821788
25%         0.000000
50%         0.000000
75%         0.000000
max        41.959513
Name: ΔFlux, dtype: float64
✅ Saved: ../data/gimme_single_results\GIMME_18C.xlsx

==== [3/323] ====

📂 Processing: 20C.csv
列名: ['Gene', '20C - ']
有效基因数: 3811
匹配成功: 1240/3779
🚀 Running GI